# Iris: an SFD, a manifest and a recoverable parameter sweep

This notebook uses the real Iris observations bundled with scikit-learn. It trains a four-candidate sweep, then pauses a manifest run after one trial, restricts its pending queue through a local intervention file and resumes through the CLI in a fresh Python process. Trained weights, caller source and scientific records remain in the printed study directory.

Use Python 3.11 or later and install the maintained generation with `python -m pip install 'talos[torch] @ git+https://github.com/autonomio/talos.git@master'` for the default backend. To use Keras or TensorFlow, replace `[torch]` in that command with `[keras,torch]` and set `KERAS_BACKEND=torch`, or use `[tensorflow]`, then change `backend` below before running the cells. Set `KERAS_BACKEND` before importing standalone Keras when choosing its execution backend. Run the cells in order in a fresh kernel. The short training budget demonstrates the contracts; it does not establish a research-quality performance estimate.


In [ ]:
import hashlib
import json
import os
import subprocess
import sys
import tempfile
from importlib.resources import files
from pathlib import Path

import numpy as np
import yaml
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split

from talos import run
from talos.experiment import RunResult
from talos.experiment.runner import load_sfd
from talos.yaml.compiler import CompiledSFD
from talos.yaml.store import canonical_manifest_id

backend = os.environ.get('TALOS_EXAMPLE_BACKEND', 'torch')  # Or select 'keras' / 'tensorflow'.
if backend == 'torch':
    import torch
    torch.set_num_threads(1)
elif backend == 'tensorflow':
    import tensorflow
elif backend == 'keras':
    import keras
else:
    raise ValueError('Choose torch, keras or tensorflow')

study = Path(tempfile.mkdtemp(prefix='talos-iris-sfd-'))
print('Retained study:', study)


## Caller-owned data and a fixed held-out split

The caller creates the data archive. Talos does not load a dataset or choose a reader. Both Python and CLI execution use these same 120 training rows and 30 held-out rows. Preparation fits scaling only on the training partition.


In [ ]:
iris = load_iris()
train_rows, val_rows = train_test_split(
    np.arange(len(iris.target)), test_size=.2, stratify=iris.target, random_state=17)
caller_data = {'x_train': iris.data[train_rows], 'y_train': iris.target[train_rows],
               'x_val': iris.data[val_rows], 'y_val': iris.target[val_rows]}
assert len(train_rows) == 120 and len(val_rows) == 30
assert not set(train_rows) & set(val_rows)
assert np.bincount(caller_data['y_train']).tolist() == [40, 40, 40]
assert np.bincount(caller_data['y_val']).tolist() == [10, 10, 10]
data_path = study / 'iris_splits.npz'
np.savez(data_path, **caller_data, train_rows=train_rows, val_rows=val_rows)
print({'dataset': 'sklearn bundled Iris', 'training_rows': len(train_rows),
       'held_out_rows': len(val_rows),
       'data_sha256': hashlib.sha256(data_path.read_bytes()).hexdigest()})


## Ordinary Python: `params`, `prep` and `model`

Copy the selected packaged framework template into caller-owned source. Keep its native training loop and artifact factory. The appended functions define this study's parameter domains, load caller data when invoked through the CLI, fit train-only scaling and report held-out log loss and accuracy. The framework template stays editable in the study directory.


In [ ]:
template_name = {'torch': 'pytorch', 'keras': 'keras', 'tensorflow': 'tf_keras'}[backend]
template = files('talos.sfd.templates').joinpath(template_name + '.py').read_text()
sfd_path = study / 'iris_sfd.py'
extension = """

# Study-specific SFD functions; the native template owns model construction/training.
_template_model = model


def params():
    return {'units': [8, 16], 'learning_rate': [.01, .03],
            'epochs': [15], 'batch_size': [16]}


def prep(context, round_params):
    import numpy as np
    from sklearn.preprocessing import StandardScaler
    if 'data_path' in context:
        with np.load(context['data_path'], allow_pickle=False) as archive:
            data = {key: archive[key] for key in ('x_train', 'y_train', 'x_val', 'y_val')}
    else:
        data = dict(context)
    scaler = StandardScaler().fit(data['x_train'])
    return {**data, 'x_train': scaler.transform(data['x_train']),
            'x_val': scaler.transform(data['x_val']), 'task': 'multiclass'}


def probabilities(network, x):
    import numpy as np
    from talos.backends import backend_for
    values = np.asarray(backend_for(network, backend=backend).predict(network, x))
    if backend == 'torch':
        values = np.exp(values - values.max(axis=1, keepdims=True))
        values /= values.sum(axis=1, keepdims=True)
    return values


def model(prepared, round_params):
    from sklearn.metrics import accuracy_score, log_loss
    history, network = _template_model(prepared, round_params)
    prediction = probabilities(network, prepared['x_val'])
    return {'model': network, 'history': history, 'backend': backend,
            'val_loss': float(log_loss(prepared['y_val'], prediction, labels=[0, 1, 2])),
            'val_accuracy': float(accuracy_score(prepared['y_val'], prediction.argmax(axis=1)))}
"""
sfd_path.write_text(template + extension)
sfd = load_sfd(sfd_path)
prepared = sfd.prep(caller_data, {})
np.testing.assert_allclose(prepared['x_train'].mean(axis=0), 0, atol=1e-12)
print({'sfd': str(sfd_path), 'backend': sfd.backend, 'params': sfd.params()})


## Run the complete sweep with caller data

`run` passes the caller dictionary to `prep`. Four unique combinations must complete. The objective explicitly minimizes held-out log loss. Accuracy remains a separate reported metric.


In [ ]:
native = run(sfd_path, data=caller_data, seed=17, progress_bar=False,
             experiment_dir=study / 'python_run',
             objective={'metric': 'val_loss', 'direction': 'min'})
expected_combinations = {(units, rate) for units in (8, 16) for rate in (.01, .03)}
assert native.status == 'complete' and len(native.data) == 4
assert set(zip(native.data['units'], native.data['learning_rate'])) == expected_combinations
assert native.data['_trial_id'].nunique() == 4
assert np.isfinite(native.data['val_loss']).all()
assert native.data['val_accuracy'].between(0, 1).all()
print(native.data[['units', 'learning_rate', 'val_loss', 'val_accuracy']].to_string(index=False))


## Compile the same SFD into a manifest

The manifest records the backend, explicit objective, parameter overrides, seed, strategy and artifact policy. Its `context` contains the caller archive path; the SFD's own `prep` loads it. Only the epoch domain changes here, to illustrate an override. No remote service is involved.


In [ ]:
document = {
    'schema_version': '1.0',
    'metadata': {'name': 'iris_sfd_manifest', 'mode': 'development'},
    'sfd': {'module': sfd_path.name, 'backend': backend,
            'objective': {'metric': 'val_loss', 'direction': 'min'},
            'context': {'data_path': str(data_path)}, 'params': {'epochs': [10]}},
    'uel': {'seed': 17, 'round_limit': 4, 'search_strategy': {'type': 'grid'},
            'checkpoint_interval': 1, 'feedback_interval': 1,
            'save_models': True, 'progress_bar': False}}
manifest_path = study / 'iris.yaml'
manifest_path.write_text(yaml.safe_dump(document, sort_keys=False))
compiled = CompiledSFD(document, source_path=manifest_path)
assert compiled.params()['epochs'] == [10]
print('Manifest identity:', canonical_manifest_id(document))
print(manifest_path.read_text())


The CLI runs under this kernel's interpreter. Validation checks configuration; dry-run also resolves the SFD and search space. Neither trains a model.


In [ ]:
commands = []


def cli(*arguments):
    command = [sys.executable, '-m', 'talos', *map(str, arguments)]
    completed = subprocess.run(command, cwd=study, capture_output=True, text=True, check=True)
    commands.append({'arguments': list(map(str, arguments)), 'returncode': completed.returncode,
                     'stdout': completed.stdout, 'stderr': completed.stderr})
    print(completed.stdout, end='')
    if completed.stderr:
        print(completed.stderr, end='')
    return completed


cli('validate', manifest_path)
cli('run', '--dry-run', manifest_path)
assert not (study / 'results').exists()


## Pause after one committed trial

The compiled manifest executes through the same native runner. `stop_after=1` pauses between trials. The checkpoint and saved weights represent the one completed trial; unfinished epochs are outside this recovery boundary.


In [ ]:
paused = compiled.execute(experiment_dir=study / 'manifest_run', stop_after=1)
assert paused.status == 'paused' and len(paused.data) == 1
run_dir = paused.run_dir
first_record = (run_dir / 'round_data.jsonl').read_text().splitlines()[0]
first_artifact = run_dir / paused.artifacts[0]['path']
first_weights = (hashlib.sha256(first_artifact.read_bytes()).hexdigest(),
                 first_artifact.stat().st_mtime_ns)
identity = paused.metadata['identity_hash']
assert (run_dir / 'checkpoint.json').is_file()
assert (run_dir / 'metadata.json').is_file()
print({'status': paused.status, 'completed_trials': len(paused.data),
       'first_params': paused._records[0]['params'], 'run_dir': str(run_dir)})


## Control pending work and resume in a fresh process

Write one local intervention to keep eight-unit candidates. It applies to pending work and records an audit entry. The first completed trial remains intact; one additional candidate trains. Resuming again after completion must leave both records and weights unchanged.


In [ ]:
intervention = [{'op': 'keep_is', 'param': 'units', 'value': 8}]
(run_dir / 'interventions.json').write_text(json.dumps(intervention))
cli('run', '--resume', run_dir, '--no-progress-bar')
recovered = RunResult.load(run_dir)
assert recovered.status == 'complete' and len(recovered.data) == 2
assert list(zip(recovered.data['units'], recovered.data['learning_rate'])) == [(8, .01), (8, .03)]
assert recovered.data['_trial_id'].nunique() == 2
assert recovered.metadata['identity_hash'] == identity
assert (run_dir / 'round_data.jsonl').read_text().splitlines()[0] == first_record
assert (hashlib.sha256(first_artifact.read_bytes()).hexdigest(), first_artifact.stat().st_mtime_ns) == first_weights
audit = [json.loads(line) for line in (run_dir / 'audit.jsonl').read_text().splitlines()]
assert all(not entry['errors'] for entry in audit)
assert any(item['op'] == 'keep_is' and item['source'] == 'intervention_file'
           for entry in audit for item in entry['interventions'])
print(recovered.data[['units', 'learning_rate', 'val_loss', 'val_accuracy']].to_string(index=False))
print('Audited intervention:', intervention)
records_before = (run_dir / 'round_data.jsonl').read_bytes()
cli('run', '--resume', run_dir, '--no-progress-bar')
assert (run_dir / 'round_data.jsonl').read_bytes() == records_before
assert (hashlib.sha256(first_artifact.read_bytes()).hexdigest(), first_artifact.stat().st_mtime_ns) == first_weights


## Restore trained artifacts and check the recorded metric

Load every saved model and recompute its held-out log loss and accuracy. These must agree with the recorded values. The automatic best model must minimize `val_loss`, even though accuracy is also present. Restoration loads weights; it does not retrain. The separate train-only scaler is reconstructed by this SFD's verified `prep`; it is not embedded in the network artifact.


In [ ]:
from talos.utils.best_model import activate_model

held_out = sfd.prep({'data_path': str(data_path)}, {})
for index, row in recovered.data.iterrows():
    prediction = sfd.probabilities(activate_model(recovered, index), held_out['x_val'])
    assert prediction.shape == (30, 3)
    np.testing.assert_allclose(prediction.sum(axis=1), 1, atol=1e-6)
    np.testing.assert_allclose(log_loss(held_out['y_val'], prediction, labels=[0, 1, 2]),
                               row['val_loss'], rtol=1e-6, atol=1e-7)
    np.testing.assert_allclose(accuracy_score(held_out['y_val'], prediction.argmax(axis=1)),
                               row['val_accuracy'], atol=1e-12)
best_prediction = sfd.probabilities(recovered.best_model(), held_out['x_val'])
np.testing.assert_allclose(log_loss(held_out['y_val'], best_prediction, labels=[0, 1, 2]),
                           recovered.data['val_loss'].min(), rtol=1e-6, atol=1e-7)
assert (run_dir / 'round_data.jsonl').read_bytes() == records_before
print({'best_val_loss': float(recovered.data['val_loss'].min()),
       'prediction_shape': list(best_prediction.shape), 'restored_without_retraining': True})


## Keep the evidence

The study directory contains the caller data archive and SFD, editable YAML, complete Python sweep, manifest checkpoint, source snapshots, audit trail and native trained artifacts. Keep these with the recorded backend/dependency environment. A manifest identifies configuration; the run records additionally attest source, data, trial identities and artifacts. Resume rejects changed verified inputs. This example makes no cross-version checkpoint migration claim.


In [ ]:
evidence = {
    'dataset': 'sklearn bundled Iris', 'backend': backend,
    'data_sha256': hashlib.sha256(data_path.read_bytes()).hexdigest(),
    'sfd_sha256': hashlib.sha256(sfd_path.read_bytes()).hexdigest(),
    'manifest_id': canonical_manifest_id(document), 'run_identity': identity,
    'native_trials': len(native.data), 'recovered_trials': len(recovered.data),
    'parameter_combinations': recovered.data[['units', 'learning_rate']].values.tolist(),
    'metrics': recovered.data[['val_loss', 'val_accuracy']].to_dict('records'),
    'completed_first_trial_preserved': True, 'commands': commands,
    'environment': recovered.metadata['environment'],
    'artifacts': recovered.artifacts,
    'study_directory': str(study)}
(study / 'example_evidence.json').write_text(json.dumps(evidence, indent=2) + '\n')
print('Evidence:', study / 'example_evidence.json')
print('Retained native model files:', [str(run_dir / item['path']) for item in recovered.artifacts])
